# UAEFA API Probe

Notebook de exploracion prudente para trabajar con endpoints reales de `https://www.uaefa.ae/api/...` sin hacer scraping agresivo.

## Objetivo

- Validar endpoints reales confirmados desde DevTools
- Reutilizar ids y parametros vistos en respuestas buenas
- Hacer pocas peticiones, con cache y pausas aleatorias
- Evitar cookies copiadas a mano mientras no sean necesarias
- Construir una base elegante para seguir con clubes, competiciones y partidos

## Principios de prudencia

- Nada de barridos masivos ni adivinacion de endpoints
- Solo usamos rutas confirmadas por el propio navegador
- Cada respuesta se cachea para no repetir trafico
- El ritmo de peticiones es deliberadamente lento
- Si una llamada no aporta valor, no se repite

## 1. Preparacion del entorno

Cargamos utilidades locales y preparamos una sesion HTTP ligera, con cabeceras minimas y realistas.

In [40]:
from pathlib import Path
import json
import sys

# Configuracion de rutas del proyecto.
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from uaefa_scraper.client import GentleClient

# Directorios para respuestas crudas y resumentes derivados.
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROBE_DIR = RAW_DIR / 'api_probe'
PROBE_DIR.mkdir(parents=True, exist_ok=True)

# Cliente conservador: pausas largas y cache activada.
client = GentleClient(
    cache_dir=PROBE_DIR / 'cache',
    min_delay=1.5,
    max_delay=3.0,
    timeout=30.0,
)

# Cabeceras minimas inspiradas en las peticiones del navegador.
client.session.headers.update({
    'Accept': 'application/json, text/plain, */*',
    'Referer': 'https://www.uaefa.ae/en/competitions',
})

PROJECT_ROOT

PosixPath('/Users/macmontxinho/Desktop/Teams/Scraping UAE')

## 2. Endpoint semilla: competiciones

Empezamos por la llamada confirmada en `DevTools`. Esta respuesta nos aporta ids reales de competicion, ronda y grupo.

In [41]:
# Endpoint confirmado por el navegador para la pagina de competiciones.
competitions_url = 'https://www.uaefa.ae/api/competitions?lang=en'
competitions_payload = client.fetch_json(competitions_url)

# Resumen corto para validar que la llamada sigue viva y estable.
{
    'error': competitions_payload.get('error'),
    'msg': competitions_payload.get('msg'),
    'season': competitions_payload.get('season'),
    'competition_count': len(competitions_payload.get('competitions', [])),
}

{'error': False,
 'msg': 'Competitions listed successfully',
 'season': {'season_id': '59', 'name': '2026/2027'},
 'competition_count': 53}

## 3. Normalizacion ligera de competiciones

La respuesta puede traer `round` y `group` como objeto o lista. Normalizamos esa parte para dejar una estructura plana reutilizable.

In [42]:
def ensure_list(value):
    """Convierte un valor en lista para homogeneizar respuestas mixtas."""
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]


def normalize_competitions(payload):
    """Extrae una tabla simple de competiciones, rondas y grupos."""
    rows = []
    for competition in payload.get('competitions', []):
        comp_attr = competition.get('@attributes', {})
        rounds = ensure_list(competition.get('round'))
        for round_item in rounds:
            round_attr = round_item.get('@attributes', {}) if isinstance(round_item, dict) else {}
            groups = ensure_list(round_item.get('group') if isinstance(round_item, dict) else None)
            if not groups:
                rows.append({
                    'competition_id': comp_attr.get('competition_id'),
                    'competition_name': comp_attr.get('name'),
                    'round_id': round_attr.get('round_id'),
                    'round_name': round_attr.get('name'),
                    'group_id': None,
                    'group_name': None,
                    'stage': None,
                })
                continue
            for group_item in groups:
                group_attr = group_item.get('@attributes', {}) if isinstance(group_item, dict) else {}
                rows.append({
                    'competition_id': comp_attr.get('competition_id'),
                    'competition_name': comp_attr.get('name'),
                    'round_id': round_attr.get('round_id'),
                    'round_name': round_attr.get('name'),
                    'group_id': group_attr.get('group_id'),
                    'group_name': group_attr.get('name'),
                    'stage': group_attr.get('stage'),
                })
    return rows


competition_rows = normalize_competitions(competitions_payload)
len(competition_rows)

124

In [43]:
# Vista rapida de las primeras filas normalizadas.
competition_rows[:10]

[{'competition_id': '2570',
  'competition_name': 'President Cup',
  'round_id': '4688',
  'round_name': 'Round 16',
  'group_id': '8532',
  'group_name': 'One Group',
  'stage': 'One Round'},
 {'competition_id': '2570',
  'competition_name': 'President Cup',
  'round_id': '4689',
  'round_name': 'Round 4',
  'group_id': '8533',
  'group_name': 'One Group',
  'stage': 'One Round'},
 {'competition_id': '2570',
  'competition_name': 'President Cup',
  'round_id': '4690',
  'round_name': 'Round 16 for preliminary qualifiers for first-division clubs',
  'group_id': '8534',
  'group_name': 'One Group',
  'stage': 'One Round'},
 {'competition_id': '2570',
  'competition_name': 'President Cup',
  'round_id': '4691',
  'round_name': 'Round 8',
  'group_id': '8535',
  'group_name': 'One Group',
  'stage': 'One Round'},
 {'competition_id': '2570',
  'competition_name': 'President Cup',
  'round_id': '4692',
  'round_name': 'Final Match',
  'group_id': '8536',
  'group_name': 'One Group',
  'stag

## 4. Endpoint confirmado: partidos por fecha

Esta llamada viene de la pagina `matches` y usa un rango de fechas. Empezamos con una sola fecha para mantener la prueba pequena.

In [44]:
# Parametros de prueba minimos para la pagina de partidos.
match_date_from = '2026-07-01'
match_date_to = '2026-07-01'

# Ajustamos el referer para parecer la navegacion natural de la pagina de partidos.
client.session.headers['Referer'] = 'https://www.uaefa.ae/en/matches'

matches_url = (
    'https://www.uaefa.ae/api/matches?lang=en'
    f'&date_from={match_date_from}'
    f'&date_to={match_date_to}'
)

matches_payload = client.fetch_json(matches_url)

# Resumen corto para entender la forma general de la respuesta.
{
    'payload_type': type(matches_payload).__name__,
    'top_level_keys': list(matches_payload.keys())[:15] if isinstance(matches_payload, dict) else None,
    'sample_size': len(matches_payload) if isinstance(matches_payload, list) else None,
}

{'payload_type': 'dict',
 'top_level_keys': ['error', 'msg', 'matches', 'active_season'],
 'sample_size': None}

In [45]:
# Muestra corta de la respuesta de partidos para ver su estructura sin imprimirlo todo.
matches_payload if isinstance(matches_payload, dict) else matches_payload[:3]

{'error': False,
 'msg': 'Matches listed successfully',
 'matches': [],
 'active_season': '59'}

## 5. Seleccion de ids semilla

Guardamos un ejemplo real de competicion normalizada para usarlo despues si descubrimos endpoints de detalle o clasificacion.

In [46]:
# Fila real de ejemplo que podremos reutilizar en notebooks posteriores.
sample_row = competition_rows[0]
sample_row

{'competition_id': '2570',
 'competition_name': 'President Cup',
 'round_id': '4688',
 'round_name': 'Round 16',
 'group_id': '8532',
 'group_name': 'One Group',
 'stage': 'One Round'}

## 6. Catalogo controlado de endpoints confirmados

Solo registramos aqui endpoints realmente vistos en DevTools. Los placeholders sirven para ir completando el mapa sin inventar rutas.

In [47]:
# Mapa pequeno de endpoints confirmados o pendientes de confirmar.
api_candidates = [
    {
        'name': 'competitions',
        'url': competitions_url,
        'status': 'confirmed',
    },
    {
        'name': 'matches_by_date',
        'url': matches_url,
        'status': 'confirmed',
    },
    {
        'name': 'club_placeholder',
        'url': None,
        'status': 'pending_from_devtools',
    },
]

api_candidates

[{'name': 'competitions',
  'url': 'https://www.uaefa.ae/api/competitions?lang=en',
  'status': 'confirmed'},
 {'name': 'matches_by_date',
  'url': 'https://www.uaefa.ae/api/matches?lang=en&date_from=2026-07-01&date_to=2026-07-01',
  'status': 'confirmed'},
 {'name': 'club_placeholder', 'url': None, 'status': 'pending_from_devtools'}]

## 7. Resumen ligero de respuestas cacheadas

Guardamos un resumen pequeno de los endpoints confirmados para revisarlos sin reconsultar la API.

In [48]:
def build_payload_summary(name, url, payload):
    """Crea un resumen corto y estable de una respuesta ya obtenida."""
    payload_text = json.dumps(payload, ensure_ascii=False)
    if isinstance(payload, dict):
        top_level_keys = list(payload.keys())[:15]
    else:
        top_level_keys = None
    return {
        'name': name,
        'url': url,
        'payload_type': type(payload).__name__,
        'top_level_keys': top_level_keys,
        'snippet': payload_text[:400],
    }


probe_results = [
    build_payload_summary('competitions', competitions_url, competitions_payload),
    build_payload_summary('matches_by_date', matches_url, matches_payload),
]

probe_results

[{'name': 'competitions',
  'url': 'https://www.uaefa.ae/api/competitions?lang=en',
  'payload_type': 'dict',
  'top_level_keys': ['error', 'msg', 'season', 'competitions'],
  'snippet': '{"error": false, "msg": "Competitions listed successfully", "season": {"season_id": "59", "name": "2026/2027"}, "competitions": [{"@attributes": {"competition_id": "2570", "name": "President Cup", "isPLC": "0"}, "round": [{"@attributes": {"round_id": "4688", "name": "Round 16", "no_order": "0"}, "group": {"@attributes": {"group_id": "8532", "name": "One Group", "stage": "One Round"}}}, {"@attribut'},
 {'name': 'matches_by_date',
  'url': 'https://www.uaefa.ae/api/matches?lang=en&date_from=2026-07-01&date_to=2026-07-01',
  'payload_type': 'dict',
  'top_level_keys': ['error', 'msg', 'matches', 'active_season'],
  'snippet': '{"error": false, "msg": "Matches listed successfully", "matches": [], "active_season": "59"}'}]

In [49]:
# Persistimos resumenes y datos normalizados para no repetir procesamiento.
probe_summary_path = PROBE_DIR / 'confirmed_api_summaries.json'
probe_summary_path.write_text(json.dumps(probe_results, ensure_ascii=False, indent=2), encoding='utf-8')

normalized_path = PROBE_DIR / 'competitions_normalized.json'
normalized_path.write_text(json.dumps(competition_rows, ensure_ascii=False, indent=2), encoding='utf-8')

{
    'probe_summary_path': str(probe_summary_path),
    'normalized_path': str(normalized_path),
}

{'probe_summary_path': '/Users/macmontxinho/Desktop/Teams/Scraping UAE/data/raw/api_probe/confirmed_api_summaries.json',
 'normalized_path': '/Users/macmontxinho/Desktop/Teams/Scraping UAE/data/raw/api_probe/competitions_normalized.json'}

## 8. Siguiente descubrimiento de bajo ruido

El siguiente paso recomendable es capturar la URL real del endpoint de una ficha de club o del detalle de una competicion. Solo hace falta copiar la URL y el inicio del JSON.

In [50]:
# Checklist corto para la proxima captura en DevTools.
next_step_checklist = {
    'copy_from_devtools': [
        'Request URL',
        'Request Method',
        'Status Code',
        'content-type',
        'inicio del JSON de respuesta',
    ],
    'priority_pages': [
        'https://www.uaefa.ae/en/club/1',
        'https://www.uaefa.ae/en/competition/2571',
    ],
}

next_step_checklist

{'copy_from_devtools': ['Request URL',
  'Request Method',
  'Status Code',
  'content-type',
  'inicio del JSON de respuesta'],
 'priority_pages': ['https://www.uaefa.ae/en/club/1',
  'https://www.uaefa.ae/en/competition/2571']}